In [10]:
import os
import platform
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from multiprocessing import cpu_count

from model.proposed import MYModule

### 可复现性

In [11]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
pl.seed_everything(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Global seed set to 42


### hyperparameter tuning

In [12]:
SEQ_LEN = 200
BATCH_SIZE = 64
EMBED_DIM = 128
DROP_OUT = 0.9
NUM_WORKERS = 0 if platform.system() == 'Windows' else cpu_count()
N_GCN_LAYERS = 5
Q_is_S = False
print("os:{}, num-workers:{}".format(platform.system(), NUM_WORKERS))

os:Linux, num-workers:16


### load assist data

In [13]:
dataset_name = 'assist09'
dataset_path = os.path.join(os.getcwd(), 'dataset', dataset_name)

df = pd.read_csv(os.path.join(dataset_path, "processed_data.csv"), low_memory=False, encoding="ISO-8859-1")

key_user = df.columns[0]
key_q = df.columns[1]
key_s = df.columns[2]
key_qtype =df.columns[3]
key_ms = df.columns[4]
key_attempt = df.columns[5]
key_correct = df.columns[6]
key_diff = df.columns[7]

N_QUESTION = len(df[key_q].unique())
N_SKILL = len(df[key_s].unique())
N_QUESTION_TYPE = len(df[key_qtype].unique())

print("num of question:{}, num of skill:{}, n_question_type:{}".format(N_QUESTION, N_SKILL, N_QUESTION_TYPE))
print(df.columns)

num of question:16891, num of skill:101, n_question_type:5
Index(['user_id', 'problem_id', 'skill_name', 'answer_type',
       'ms_first_response', 'attempt_count', 'correct', 'difficulty'],
      dtype='object')


In [14]:
# 我们需要将数据进行预处理，每个学生的学习记录利用group by合并为序列。
def generate_group_by_df(df):
    KEY = key_s if Q_is_S else key_q
    group = df.groupby([key_user]).apply(lambda r: (
                r[KEY].values,
                r[key_s].values,
                r[key_qtype].values,
                r[key_diff].values,
                r[key_ms].values,
                r[key_attempt].values,
                r[key_correct].values                                                                                                                                                                                                                                      
                ))
    return group



# df_train = pd.read_csv(os.path.join(dataset_path, "train.csv"), low_memory=False, encoding="ISO-8859-1")
# df_test = pd.read_csv(os.path.join(dataset_path, "test.csv"), low_memory=False, encoding="ISO-8859-1")
# train, val = generate_group_by_df(df_train), generate_group_by_df(df_test)


group = generate_group_by_df(df)
train, val = train_test_split(group, test_size=0.2)
len_list = [len(group.iloc[i][0]) for i in range(len(group))]

N_QUERY_FEATURES = len(train.iloc[0])-1
print("N_QUERY_FEATURES:{}".format(N_QUERY_FEATURES))
print("seq_len mean:{}, max:{}, min:{}".format(np.mean(len_list), np.max(len_list), np.min(len_list)))

N_QUERY_FEATURES:6
seq_len mean:66.15032522283786, max:1040, min:1


###  assist09 dataset

In [15]:
from data_loader.assist09 import Assist09Dataset
from data_loader.saintdataset import SAINTDataset

N_Q_OR_S = N_SKILL if Q_is_S else N_QUESTION


train_dataset = SAINTDataset(train, N_Q_OR_S, SEQ_LEN)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

val_dataset = SAINTDataset(val, N_Q_OR_S, SEQ_LEN)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("train:{}, test:{}".format(len(train_dataset), len(val_dataset)))

train:3320, test:831


In [16]:
train_dataset[0][2]

array([0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0,
       1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0,
       0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1,
       1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0,
       0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0,
       1])

In [17]:
import warnings
warnings.filterwarnings('ignore')

from model.saint2 import SAINTModule


model = SAINTModule(
    dim_model=EMBED_DIM,
    num_en=6,
    num_de=6,
    heads_en=8,
    heads_de=8,
    total_ex=N_Q_OR_S,
    total_cat=N_QUESTION_TYPE,
    total_in=2,
    seq_len=SEQ_LEN
)
checkpoint_callback = pl.callbacks.ModelCheckpoint(save_top_k=1, verbose=True, monitor='v_auc', mode='max')

patience = 6 if N_QUERY_FEATURES == 6 else 3
# sakt.train_dataloader
trainer = pl.Trainer(
    gpus=1, 
    max_epochs=200, 
    auto_lr_find=True, 
    callbacks=[checkpoint_callback, EarlyStopping(monitor="v_auc", mode="max", patience=6)]
)
print("patience:{}".format(patience))


GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


patience:6


In [18]:
trainer.fit(model=model, train_dataloaders=train_dataloader,val_dataloaders=val_dataloader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type              | Params
--------------------------------------------
0 | loss  | BCEWithLogitsLoss | 0     
1 | model | saint             | 14.6 M
--------------------------------------------
14.6 M    Trainable params
0         Non-trainable params
14.6 M    Total params
58.282    Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Epoch 0, global step 52: 'v_auc' reached 0.60945 (best 0.60945), saving model to '/home/czy/KT/BRIKT/lightning_logs/version_40/checkpoints/epoch=0-step=52.ckpt' as top 1


...

Epoch 4, global step 260: 'v_auc' was not in top 1


Validation: 0it [00:00, ?it/s]

Epoch 5, global step 312: 'v_auc' was not in top 1


Validation: 0it [00:00, ?it/s]

Epoch 6, global step 364: 'v_auc' was not in top 1
